# Прототип DeepCluster: проверка данных и инфраструктуры обучения

Этот ноутбук продолжает технический прототип: повторно восстанавливает encoder, проверяет связь текстов с embedding-полем parquet и подготавливает датасет для псевдометок.

Рабочий масштаб прототипа — до 500 000 товаров с сохранением категориального распределения. Он полезен для проверки производительности, но не применяется для финального обучения на Mac: для воспроизводимого эксперимента с KNN-валидацией используется 20 000 товаров train и 5 000 validation в ноутбуке 8.

In [10]:
import os
import json
import numpy as np
import pandas as pd
import torch

from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

# from transformers import BertTokenizer

from pathlib import Path

LOCAL_DATA_DIR = Path("/Users/denis/Downloads")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = str(LOCAL_DATA_DIR / "dataset-electronics-5M.parquet")
CHECKPOINT_PATH = str(LOCAL_DATA_DIR / "rubert-tiny2-custom.pth")
TOKENIZER_DIR = "/Users/denis/Desktop/coursework/Course_work_community_detection/models/rubert-tiny2-tokenizer"
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
RANDOM_STATE = 42

print("Device:", DEVICE)
print("Dataset:", DATA_PATH)
print("Checkpoint:", CHECKPOINT_PATH)
print("Tokenizer:", TOKENIZER_DIR)

Device: mps
Checkpoint: /Users/denis/Desktop/coursework/Course_work_community_detection/models/epoch_64_encoder.pth
Tokenizer: /Users/denis/Desktop/coursework/Course_work_community_detection/models/rubert-tiny2-tokenizer


In [13]:
from transformers import BertTokenizer

In [14]:
tokenizer = BertTokenizer.from_pretrained(
    TOKENIZER_DIR,
    local_files_only=True
)

print("Tokenizer size:", len(tokenizer))
print(tokenizer.tokenize("Смартфон Apple iPhone 15 128GB"))

Tokenizer size: 83828
['Смарт', '##фон', 'Apple', 'iPhone', '15', '128', '##GB']


In [15]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

print(checkpoint.keys())
print(checkpoint["meta"])

dict_keys(['meta', 'state_dict', 'optimizer', 'scheduler'])
{'epoch': 64, 'iter': 6400, 'time': 'Mon Sep  8 11:22:19 2025'}


## Загрузка модели и контроль размерностей

Архитектура и checkpoint проверяются до запуска кластеризации. Это важно, поскольку совместимость слоёв, токенизатора и веса encoder определяет, будут ли новые CLS-векторы сопоставимы с исходными эмбеддингами товаров.

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


class BertLayer(nn.Module):
    def __init__(
        self,
        hidden_size=312,
        num_heads=12,
        intermediate_size=600
    ):
        super().__init__()

        self.attention = BertAttention(
            hidden_size,
            num_heads
        )

        self.intermediate = nn.Module()
        self.intermediate.dense = nn.Linear(
            hidden_size,
            intermediate_size
        )

        self.output = BertOutput(
            intermediate_size,
            hidden_size
        )

    def forward(self, x, attention_mask):
        x = self.attention(x, attention_mask)

        intermediate = F.gelu(
            self.intermediate.dense(x)
        )

        x = self.output(
            intermediate,
            x
        )

        return x

class BertAttention(nn.Module):
    def __init__(
        self,
        hidden_size,
        num_heads
    ):
        super().__init__()

        self.self = BertSelfAttention(
            hidden_size,
            num_heads
        )

        self.output = BertSelfOutput(
            hidden_size
        )

    def forward(self, x, attention_mask):
        attention_output = self.self(
            x,
            attention_mask
        )

        return self.output(
            attention_output,
            x
        )


class BertSelfAttention(nn.Module):
    def __init__(
        self,
        hidden_size,
        num_heads
    ):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.query = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.key = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.value = nn.Linear(
            hidden_size,
            hidden_size
        )

    def forward(self, x, attention_mask):
        batch_size, seq_len, hidden_size = x.shape

        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        q = q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        scores = torch.matmul(
            q,
            k.transpose(-1, -2)
        )

        scores = scores / np.sqrt(self.head_dim)

        scores = scores + attention_mask

        attention_probs = torch.softmax(
            scores,
            dim=-1
        )

        context = torch.matmul(
            attention_probs,
            v
        )

        context = context.transpose(
            1,
            2
        ).contiguous()

        context = context.view(
            batch_size,
            seq_len,
            hidden_size
        )

        return context


class BertSelfOutput(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()

        self.dense = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.LayerNorm = nn.LayerNorm(
            hidden_size,
            eps=1e-12
        )

    def forward(self, x, residual):
        x = self.dense(x)

        return self.LayerNorm(
            x + residual
        )


class BertOutput(nn.Module):
    def __init__(
        self,
        intermediate_size,
        hidden_size
    ):
        super().__init__()

        self.dense = nn.Linear(
            intermediate_size,
            hidden_size
        )

        self.LayerNorm = nn.LayerNorm(
            hidden_size,
            eps=1e-12
        )

    def forward(self, x, residual):
        x = self.dense(x)

        return self.LayerNorm(
            x + residual
        )


class BertEncoder(nn.Module):
    def __init__(
        self,
        vocab_size=83828,
        hidden_size=312,
        num_layers=3,
        num_heads=12,
        intermediate_size=600,
        max_position_embeddings=2048
    ):
        super().__init__()

        self.embeddings = nn.Module()

        self.embeddings.word_embeddings = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.embeddings.position_embeddings = nn.Embedding(
            max_position_embeddings,
            hidden_size
        )

        self.embeddings.token_type_embeddings = nn.Embedding(
            2,
            hidden_size
        )

        self.embeddings.LayerNorm = nn.LayerNorm(
            hidden_size,
            eps=1e-12
        )

        self.encoder = nn.Module()

        self.encoder.layer = nn.ModuleList([
            BertLayer(
                hidden_size,
                num_heads,
                intermediate_size
            )
            for _ in range(num_layers)
        ])

    def forward(
        self,
        input_ids,
        attention_mask
    ):
        batch_size, seq_len = input_ids.shape

        position_ids = torch.arange(
            seq_len,
            device=input_ids.device
        ).unsqueeze(0).expand(
            batch_size,
            -1
        )

        token_type_ids = torch.zeros_like(
            input_ids
        )

        x = (
            self.embeddings.word_embeddings(input_ids)
            + self.embeddings.position_embeddings(position_ids)
            + self.embeddings.token_type_embeddings(token_type_ids)
        )

        x = self.embeddings.LayerNorm(x)

        extended_mask = attention_mask[
            :, None, None, :
        ].float()

        extended_mask = (
            1.0 - extended_mask
        ) * -10000.0

        for layer in self.encoder.layer:
            x = layer(
                x,
                extended_mask
            )

        return x

In [17]:
encoder = BertEncoder()

state_dict = checkpoint["state_dict"]

missing, unexpected = encoder.load_state_dict(
    state_dict,
    strict=False
)

print("Missing:", missing)
print("Unexpected:", unexpected)

print(
    "Parameters:",
    sum(p.numel() for p in encoder.parameters())
)

Missing: []
Unexpected: []
Parameters: 29096112


In [18]:
encoder = encoder.to(DEVICE)
encoder.eval()

text = "Смартфон Apple iPhone 15 128GB"

encoded = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

input_ids = encoded["input_ids"].to(DEVICE)
attention_mask = encoded["attention_mask"].to(DEVICE)

with torch.no_grad():
    output = encoder(
        input_ids,
        attention_mask
    )

print("Input:", input_ids.shape)
print("Output:", output.shape)

embedding = output[:, 0, :]

print("Embedding:", embedding.shape)
print("Norm:", torch.norm(embedding).item())

Input: torch.Size([1, 9])
Output: torch.Size([1, 9, 312])
Embedding: torch.Size([1, 312])
Norm: 16.20451545715332


## Проверка извлечения признаков

На небольшом наборе товаров вычисляются CLS-векторы из текста и декодируются готовые эмбеддинги parquet. Косинусная близость служит быстрой диагностикой соответствия двух способов получения признаков.

In [19]:
texts_test = []

for batch in pf.iter_batches(
    batch_size=100,
    columns=["model_text"]
):
    batch_texts = batch.column("model_text").to_pylist()

    for text in batch_texts:
        if isinstance(text, bytes):
            text = text.decode("utf-8", errors="replace")

        texts_test.append(text)

        if len(texts_test) >= 5:
            break

    if len(texts_test) >= 5:
        break

for i, text in enumerate(texts_test):
    print(f"{i}: {text[:200]}")

NameError: name 'pf' is not defined

In [ ]:
def get_embeddings(
    texts,
    batch_size=8,
    max_length=512
):
    encoder.eval()

    embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(DEVICE)
        attention_mask = encoded["attention_mask"].to(DEVICE)

        with torch.no_grad():
            output = encoder(
                input_ids,
                attention_mask
            )

        batch_embeddings = output[:, 0, :]

        embeddings.append(
            batch_embeddings.cpu().numpy()
        )

    return np.concatenate(embeddings, axis=0)

In [ ]:
test_embeddings = get_embeddings(
    texts_test,
    batch_size=5,
    max_length=512
)

print("Shape:", test_embeddings.shape)
print("First embedding:", test_embeddings[0][:10])

In [ ]:
embeddings_test_raw = []

for batch in pf.iter_batches(
    batch_size=5,
    columns=["embedding"]
):
    embeddings_test_raw.extend(
        batch.column("embedding").to_pylist()
    )
    break

print(
    "Количество:",
    len(embeddings_test_raw)
)

print(
    "Размер первого:",
    len(embeddings_test_raw[0])
)

In [ ]:
def decode_embedding(data):
    data = bytes(data)

    if len(data) != 3122:
        raise ValueError(
            f"Неожиданный размер embedding: {len(data)}"
        )

    values = np.empty(
        312,
        dtype=np.float64
    )

    for i in range(312):
        start = 2 + i * 10

        values[i] = np.frombuffer(
            data[start:start + 8],
            dtype="<f8"
        )[0]

    return values.astype(np.float32)


original_embeddings = np.stack([
    decode_embedding(x)
    for x in embeddings_test_raw
])

print(
    "Shape:",
    original_embeddings.shape
)

print(
    "First embedding:",
    original_embeddings[0][:10]
)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = np.diag(
    cosine_similarity(
        test_embeddings,
        original_embeddings
    )
)

for i, similarity in enumerate(similarities):
    print(
        f"Text {i}: cosine similarity = {similarity:.6f}"
    )

Переходим к DeepCluster

## Стратифицированный сэмпл для нагрузочной проверки

Здесь создаётся пропорциональная выборка на 500 000 строк. Объём выбран для исследования потокового чтения и подготовки DataLoader; он не является validation-набором и не используется для подбора гиперпараметров по категориям.

In [ ]:
SAMPLE_SIZE = 25_000
N_CLUSTERS = 10
MAX_LENGTH = 128
BATCH_SIZE = 32
LEARNING_RATE = 1e-5

print("Sample size:", SAMPLE_SIZE)
print("Clusters:", N_CLUSTERS)
print("Max length:", MAX_LENGTH)
print("Batch size:", BATCH_SIZE)

In [ ]:
pf = pq.ParquetFile(DATA_PATH)

category_counts = {}

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["category_id"]
):
    categories = batch.column("category_id").to_numpy()

    unique, counts = np.unique(
        categories,
        return_counts=True
    )

    for category, count in zip(unique, counts):
        category = int(category)
        category_counts[category] = (
            category_counts.get(category, 0) + int(count)
        )

print("Categories:", len(category_counts))
print("Total rows:", sum(category_counts.values()))

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

indices_by_category = {
    category: []
    for category in category_counts
}

row_start = 0

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["category_id"]
):
    categories = batch.column("category_id").to_numpy()

    for local_idx, category in enumerate(categories):
        category = int(category)

        indices_by_category[category].append(
            row_start + local_idx
        )

    row_start += len(categories)

sample_indices = []

for category, indices in indices_by_category.items():
    sample_n = round(
        SAMPLE_SIZE *
        len(indices) /
        row_start
    )

    sample_n = min(
        sample_n,
        len(indices)
    )

    selected = rng.choice(
        indices,
        size=sample_n,
        replace=False
    )

    sample_indices.extend(selected)

sample_indices = np.array(
    sample_indices,
    dtype=np.int64
)

rng.shuffle(sample_indices)

if len(sample_indices) > SAMPLE_SIZE:
    sample_indices = rng.choice(
        sample_indices,
        size=SAMPLE_SIZE,
        replace=False
    )

print("Sample:", len(sample_indices))

## Датасет для псевдометок

Каждому индексу товара соответствует текст, токенизированный до фиксированной длины, и псевдометка кластера. Индекс row groups и кэширование необходимы, чтобы DataLoader не читал parquet целиком при каждом обращении к отдельному объекту.

In [ ]:
from torch.utils.data import Dataset, DataLoader


class DeepClusterDataset(Dataset):
    def __init__(
        self,
        parquet_path,
        indices,
        pseudo_labels,
        tokenizer,
        max_length=512
    ):
        self.parquet_path = parquet_path
        self.indices = np.asarray(
            indices,
            dtype=np.int64
        )
        self.pseudo_labels = np.asarray(
            pseudo_labels,
            dtype=np.int64
        )
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.pf = pq.ParquetFile(
            parquet_path
        )

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        row_idx = self.indices[idx]

        table = self.pf.read_row_group(
            self._find_row_group(row_idx),
            columns=["model_text"]
        )

        text = table["model_text"][
            row_idx - self._row_group_start
        ].as_py()

        if isinstance(text, bytes):
            text = text.decode(
                "utf-8",
                errors="replace"
            )

        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "label": torch.tensor(
                self.pseudo_labels[idx],
                dtype=torch.long
            )
        }

Создаём индекс row groups

In [ ]:
pf = pq.ParquetFile(DATA_PATH)

row_group_starts = []
row_group_sizes = []

current_start = 0

for i in range(pf.num_row_groups):
    size = pf.metadata.row_group(i).num_rows

    row_group_starts.append(current_start)
    row_group_sizes.append(size)

    current_start += size

row_group_starts = np.array(row_group_starts)
row_group_sizes = np.array(row_group_sizes)

print("Row groups:", pf.num_row_groups)
print("Total rows:", current_start)

In [ ]:
def get_row_group(row_idx):
    group = np.searchsorted(
        row_group_starts,
        row_idx,
        side="right"
    ) - 1

    return group, row_idx - row_group_starts[group]

In [ ]:
for idx in [0, 100, 100000, 1_000_000, 4_000_000]:
    group, local_idx = get_row_group(idx)
    print(
        idx,
        "→ row_group:",
        group,
        "local:",
        local_idx
    )

In [ ]:
class DeepClusterDataset(Dataset):
    def __init__(
        self,
        parquet_path,
        indices,
        pseudo_labels,
        tokenizer,
        row_group_starts,
        max_length=512
    ):
        self.indices = np.asarray(indices)
        self.pseudo_labels = np.asarray(pseudo_labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.pf = pq.ParquetFile(parquet_path)
        self.row_group_starts = row_group_starts

        self.row_groups = np.searchsorted(
            row_group_starts,
            self.indices,
            side="right"
        ) - 1

        self.cache = {}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        row_idx = self.indices[idx]
        group = self.row_groups[idx]
        local_idx = row_idx - self.row_group_starts[group]

        if group not in self.cache:
            table = self.pf.read_row_group(
                int(group),
                columns=["model_text"]
            )
            self.cache[group] = table["model_text"]

        text = self.cache[group][local_idx].as_py()

        if isinstance(text, bytes):
            text = text.decode(
                "utf-8",
                errors="replace"
            )

        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "label": torch.tensor(
                self.pseudo_labels[idx],
                dtype=torch.long
            )
        }

In [ ]:
test_labels = np.zeros(
    len(sample_indices),
    dtype=np.int64
)

dataset_test = DeepClusterDataset(
    DATA_PATH,
    sample_indices,
    test_labels,
    tokenizer,
    row_group_starts,
    max_length=MAX_LENGTH
)

In [ ]:
item = dataset_test[0]

print("input_ids:", item["input_ids"].shape)
print("attention_mask:", item["attention_mask"].shape)
print("label:", item["label"])

In [ ]:
for idx in [0, 100, 1000, 10000, min(24_999, len(dataset_test) - 1)]:
    item = dataset_test[idx]

    print(
        idx,
        item["input_ids"].shape,
        item["attention_mask"].sum().item()
    )

Функция генерации embeddings

In [ ]:
def generate_embeddings(
    texts,
    batch_size=16,
    max_length=512
):
    encoder.eval()

    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(DEVICE)
        attention_mask = encoded["attention_mask"].to(DEVICE)

        with torch.no_grad():
            output = encoder(
                input_ids,
                attention_mask
            )

        embeddings = output[:, 0, :]
        all_embeddings.append(
            embeddings.cpu().numpy()
        )

        if start % 10000 == 0:
            print(
                f"{start:,}/{len(texts):,}"
            )

    return np.concatenate(
        all_embeddings,
        axis=0
    )

In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(
        self,
        parquet_path,
        indices,
        tokenizer,
        row_group_starts,
        max_length=512
    ):
        self.indices = np.asarray(indices)
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.pf = pq.ParquetFile(parquet_path)
        self.row_group_starts = row_group_starts

        self.row_groups = np.searchsorted(
            row_group_starts,
            self.indices,
            side="right"
        ) - 1

        self.cache = {}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        row_idx = self.indices[idx]
        group = self.row_groups[idx]

        local_idx = (
            row_idx -
            self.row_group_starts[group]
        )

        if group not in self.cache:
            table = self.pf.read_row_group(
                int(group),
                columns=["model_text"]
            )
            self.cache[group] = table["model_text"]

        text = self.cache[group][local_idx].as_py()

        if isinstance(text, bytes):
            text = text.decode(
                "utf-8",
                errors="replace"
            )

        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0)
        }

In [ ]:
embedding_dataset = EmbeddingDataset(
    DATA_PATH,
    sample_indices,
    tokenizer,
    row_group_starts,
    max_length=MAX_LENGTH
)

embedding_loader = DataLoader(
    embedding_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Dataset size:", len(embedding_dataset))

In [ ]:
MAX_LENGTH = 128
BATCH_SIZE = 32

print("Max length:", MAX_LENGTH)
print("Batch size:", BATCH_SIZE)

: 

In [ ]:
embedding_dataset = EmbeddingDataset(
    DATA_PATH,
    sample_indices,
    tokenizer,
    row_group_starts,
    max_length=MAX_LENGTH
)

embedding_loader = DataLoader(
    embedding_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Dataset size:", len(embedding_dataset))
print("Batches:", len(embedding_loader))

In [ ]:
encoder.eval()

embeddings = []

for batch_idx, batch in enumerate(embedding_loader):

    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)

    with torch.no_grad():
        output = encoder(
            input_ids,
            attention_mask
        )

    batch_embeddings = output[:, 0, :]

    embeddings.append(
        batch_embeddings.cpu().numpy()
    )

    if batch_idx % 500 == 0:
        print(
            f"{batch_idx:,}/{len(embedding_loader):,}"
        )

X_deep = np.concatenate(
    embeddings,
    axis=0
)

print("Shape:", X_deep.shape)

: 

## Место в общей работе

Ноутбук фиксирует этап подготовки крупномасштабного прототипа. Итоговые выводы о DeepCluster должны опираться на ноутбук 8: там псевдометки действительно используются для градиентного обучения encoder, а изменение качества представлений проверяется KNN-классификацией категорий на отложенной validation-выборке.